# Volume sweep analysis

Analysis of `volume_sweep_aggregate.csv` — actual output counts per strategy across `--n ∈ {5, 10, 20, 50, 80, 100}` over the 100-seed test set.

**Goals:**
1. Produce the headline per-strategy scaling curves to replace the hand-waved estimates in `generate.py` and `main.py` docstrings
2. Quantify MLM substitution's length-gated ceiling
3. Characterize where LLM paraphrasing underperforms its `--n` target
4. Audit per-transform compliance for constrained rewriting
5. Export figures and a summary CSV into `evaluation/results/` for use in `volume_sweep.md`

## Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ─── Configuration ───────────────────────────────────────────
# Point AGGREGATE_PATH at your completed run's aggregate CSV.
# RESULTS_DIR is where the notebook exports figures and the summary CSV.
AGGREGATE_PATH = Path("../runs/volume_sweep_20260416_141349/volume_sweep_aggregate.csv")
RESULTS_DIR = Path("../results/volume_sweep")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# If your aggregate predates the strategy-name rename, this will fix it on load.
# Safe no-op if the names are already correct.
NORMALIZE_STRATEGY_NAMES = {"llm": "llm_paraphrase", "mlm": "mlm_substitution"}

# Consistent colour/marker assignments used across every plot in the notebook
STRATEGY_ORDER = ["llm_paraphrase", "constrained", "mlm_substitution", "expansion"]
STRATEGY_COLORS = {
    "llm_paraphrase": "#2E86AB",
    "constrained": "#A23B72",
    "mlm_substitution": "#E07A5F",
    "expansion": "#81B29A",
}
STRATEGY_MARKERS = {
    "llm_paraphrase": "o",
    "constrained": "s",
    "mlm_substitution": "^",
    "expansion": "D",
}
LENGTH_COLORS = {"short": "#D62828", "medium": "#F77F00", "long": "#003049"}

In [ ]:
df = pd.read_csv(AGGREGATE_PATH)
df["strategy"] = df["strategy"].replace(NORMALIZE_STRATEGY_NAMES)

print(f"Loaded {len(df)} rows from {AGGREGATE_PATH.name}")
print(f"Seeds: {df['seed_id'].nunique()}")
print(f"--n values: {sorted(df['n'].unique())}")
print(f"Strategies: {sorted(df['strategy'].unique())}")
print(f"\nSeed length distribution:")
print(df.drop_duplicates("seed_id")["utterance_length"].value_counts())

---

## Section 1: Headline scaling

Per-strategy mean variant count at each `--n`. This is the table that will replace the hand-waved estimates in `generate_variants()`'s docstring.

The IQR (25th–75th percentile) band on each line captures per-seed variance — a strategy whose band widens with `--n` is one whose output varies a lot by seed type at high volume, not just at low volume.

In [ ]:
headline = df.pivot_table(index="n", columns="strategy", values="count", aggfunc="mean").round(1)
headline = headline[STRATEGY_ORDER]  # consistent column order
headline["total"] = headline.sum(axis=1).round(1)
headline

In [ ]:
# Per-strategy p25 / median / p75 — the fuller picture behind the mean
percentiles = df.groupby(["n", "strategy"])["count"].agg(
    p25=lambda x: np.percentile(x, 25),
    median="median",
    p75=lambda x: np.percentile(x, 75),
).unstack("strategy").round(1)
percentiles

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

for strat in STRATEGY_ORDER:
    sub = df[df["strategy"] == strat]
    stats = sub.groupby("n")["count"].agg(
        mean="mean",
        p25=lambda x: np.percentile(x, 25),
        p75=lambda x: np.percentile(x, 75),
    )
    ax.plot(stats.index, stats["mean"], marker=STRATEGY_MARKERS[strat],
            color=STRATEGY_COLORS[strat], label=strat, linewidth=2, markersize=7)
    ax.fill_between(stats.index, stats["p25"], stats["p75"],
                    color=STRATEGY_COLORS[strat], alpha=0.15)

ns = sorted(df["n"].unique())
ax.plot(ns, ns, linestyle=":", color="gray", alpha=0.5, label="y = n (perfect scaling)")

ax.set_xlabel("--n (variants per strategy requested)", fontsize=11)
ax.set_ylabel("Mean variants produced (per seed)", fontsize=11)
ax.set_title("Strategy scaling across --n (117 seeds, shaded = IQR)", fontsize=12)
ax.legend(loc="upper left", frameon=False)
ax.grid(True, alpha=0.3)
ax.set_xticks(ns)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "scaling_curves.png", dpi=120)
plt.show()

**What this shows:**

- **Constrained rewriting** follows the design exactly. It caps at ~14.5 when `n_per_constraint = 1` (i.e. `--n < 45`), then jumps to ~43.6 when `n_per_constraint = 3` (at `--n ≥ 50`) and stays flat from there. The ceiling is 45 (3 per transform × 15 transforms), and we get essentially all of it.
- **LLM paraphrasing and MLM substitution** both track linearly up to about `--n = 50`, then diverge from `y = x` on the aggregate. By `--n = 100` they return ~80% and ~65% of target respectively — but this aggregate picture obscures a strong length-gating effect. Sections 2 and 3 unpack it.
- **Expansion** scales cleanly on its `max(5, n // 4)` schedule.

The IQR bands widen considerably at `--n ≥ 50` for LLM paraphrasing and MLM substitution because long seeds continue to scale linearly while short seeds plateau — the aggregate mean hides two very different scaling regimes.

---

## Section 2: MLM ceiling is length-gated

MLM substitution works by masking each content word and collecting DistilBERT's top predictions. Its ceiling is bounded by *utterance length × valid candidates per position*. Short seeds simply have fewer maskable positions, so they run out earlier.

**Sample sizes:** the 117-seed set contains 63 short, 34 medium, and 20 long seeds. All three buckets now have enough coverage to support robust claims about ceiling behaviour.

In [ ]:
mlm = df[df["strategy"] == "mlm_substitution"]
mlm_by_length = mlm.pivot_table(
    index="n", columns="utterance_length", values="count", aggfunc="mean"
).round(1)[["short", "medium", "long"]]
mlm_by_length

In [ ]:
# Efficiency = actual mean count / requested --n
# Values near 1.0 mean the strategy is getting what you asked for;
# values far below 1.0 show structural ceilings.
mlm_efficiency = mlm_by_length.div(mlm_by_length.index, axis=0).round(2)
mlm_efficiency

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

for length in ["short", "medium", "long"]:
    sub = mlm[mlm["utterance_length"] == length]
    means = sub.groupby("n")["count"].mean()
    n_seeds = sub["seed_id"].nunique()
    ax.plot(means.index, means.values, marker="o", color=LENGTH_COLORS[length],
            label=f"{length} (n={n_seeds} seeds)", linewidth=2, markersize=7)

ax.plot(ns, ns, linestyle=":", color="gray", alpha=0.5, label="y = n")
ax.set_xlabel("--n", fontsize=11)
ax.set_ylabel("Mean MLM variants produced", fontsize=11)
ax.set_title("MLM substitution ceiling by utterance length", fontsize=12)
ax.legend(loc="upper left", frameon=False)
ax.grid(True, alpha=0.3)
ax.set_xticks(ns)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "mlm_ceiling_by_length.png", dpi=120)
plt.show()

**Key finding:** MLM substitution has a robust length-gated ceiling. The pattern is now supported by an adequate sample in each length bucket:

- **Short seeds (n=63):** ceiling around ~48 variants at `--n ≥ 80`. Median flat at 40.8 from `--n=80` onward.
- **Medium seeds (n=34):** ceiling around ~75 variants at `--n ≥ 80`.
- **Long seeds (n=20):** essentially unbounded within `--n ≤ 100`. Median of 100, p25 of 100, and 17 of 20 long seeds returned exactly 100 variants at `--n=100`.

This confirms the structural ceiling is a property of the utterance (positions × valid candidates per position), not of the strategy. The README's previous claim that MLM "plateaus around 30" was too low; actual ceilings depend on length and only short seeds hit the plateau aggressively.

---

## Section 3: LLM paraphrase underperformance is also length-driven

The aggregate shows LLM paraphrasing returning ~80% of target at `--n = 100`. A reasonable first guess would be that the 10-batch architecture is the culprit — but the per-length breakdown tells a different story. Short seeds underperform significantly; long seeds essentially match target. The model is honest about a real constraint on linguistic variation: you cannot produce 100 distinct natural paraphrases of "turn it up."

In [ ]:
llm = df[df["strategy"] == "llm_paraphrase"].copy()
llm["efficiency"] = llm["count"] / llm["n"]

# Fraction of (seed, --n) pairs where the model returned exactly --n variants
llm["hit_target"] = llm["count"] == llm["n"]
hit_rate = llm.groupby("n")["hit_target"].mean().round(3)
print("Fraction of seeds hitting the --n target exactly:")
hit_rate

In [ ]:
llm_by_length = llm.pivot_table(
    index="n", columns="utterance_length", values="count", aggfunc="mean"
).round(1)[["short", "medium", "long"]]
llm_by_length

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

for length in ["short", "medium", "long"]:
    sub = llm[llm["utterance_length"] == length]
    means = sub.groupby("n")["count"].mean()
    n_seeds = sub["seed_id"].nunique()
    ax.plot(means.index, means.values, marker="o", color=LENGTH_COLORS[length],
            label=f"{length} (n={n_seeds} seeds)", linewidth=2, markersize=7)

ax.plot(ns, ns, linestyle=":", color="gray", alpha=0.5, label="y = n")
ax.set_xlabel("--n", fontsize=11)
ax.set_ylabel("Mean llm_paraphrase variants produced", fontsize=11)
ax.set_title("LLM paraphrase efficiency by utterance length", fontsize=12)
ax.legend(loc="upper left", frameon=False)
ax.grid(True, alpha=0.3)
ax.set_xticks(ns)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "llm_ceiling_by_length.png", dpi=120)
plt.show()

In [ ]:
# The 10 short seeds where llm_paraphrase underperformed most at --n = 100
worst = (
    llm[llm["n"] == 100]
    .nsmallest(10, "count")
    [["seed_id", "seed", "count", "utterance_length", "syntactic_type"]]
)
worst

**Key finding:** LLM paraphrase output mirrors the MLM ceiling pattern but for a different reason — a content ceiling rather than a mechanical one. The model runs out of distinct natural paraphrases for short utterances.

- **Short seeds (n=63):** 71% mean efficiency at `--n=100` (returning ~71 of 100 requested). The model genuinely cannot produce 100 distinct natural paraphrases of a 3-word utterance.
- **Medium seeds (n=34):** ~86% mean efficiency at `--n=100`.
- **Long seeds (n=20):** ~97% mean efficiency at `--n=100`. Median count of 98, and most long seeds return ≥95 of 100 requested variants.

This isn't a bug and shouldn't be fixed. It is worth documenting: asking for 100 paraphrases of a 3-word utterance is not a reasonable request, and allo will honestly return fewer.

**Secondary observation on speech disfluencies:** the two long seeds carrying speech-disfluency tags (false start, filled pause — IDs 106 and 111) both landed near the bottom of the long-seed LLM paraphrase distribution at `--n=100`. Seed 111 returned 87 variants, the lowest of any long seed. Sample of 2 is far too small for a claim, but worth flagging as a future-work question: do speech disfluencies reduce LLM paraphrase yield when controlling for length?

---

## Section 4: Constrained rewriting — per-transform compliance

At low `--n` (5, 10, 20), `n_per_constraint = 1` — one variant per transform. Every transform in the current prompt set achieves 100% compliance at this setting.

At high `--n` (50, 80, 100), `n_per_constraint = 3` — three variants per transform. Compliance drops for specific transforms, and the pattern is worth looking at.

In [ ]:
# Parse the JSON blob in the constrained_by_transform column into long format
constrained = df[
    (df["strategy"] == "constrained") & df["constrained_by_transform"].notna()
].copy()

transform_rows = []
for _, row in constrained.iterrows():
    transforms = json.loads(row["constrained_by_transform"])
    for tname, tcount in transforms.items():
        transform_rows.append({
            "seed_id": row["seed_id"],
            "seed": row["seed"],
            "utterance_length": row["utterance_length"],
            "syntactic_type": row["syntactic_type"],
            "n": row["n"],
            "transform": tname,
            "count": tcount,
        })
transform_df = pd.DataFrame(transform_rows)
print(f"Parsed {len(transform_df)} (seed, n, transform) rows")

In [ ]:
# Compliance rate at high --n (n_per_constraint=3, target count == 3 per transform)
high = transform_df[transform_df["n"].isin([50, 80, 100])]
compliance_high = (
    high.groupby("transform")
    .agg(total=("count", "size"), compliant=("count", lambda x: (x == 3).sum()))
    .assign(rate=lambda d: d["compliant"] / d["total"])
    .sort_values("rate")
)
compliance_high.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(compliance_high.index, compliance_high["rate"], color="#A23B72", alpha=0.85)

# Highlight underperformers (red <80%, orange <95%)
for i, (_, row) in enumerate(compliance_high.iterrows()):
    if row["rate"] < 0.80:
        bars[i].set_color("#D62828")
    elif row["rate"] < 0.95:
        bars[i].set_color("#F77F00")

ax.set_xlim(0, 1.05)
ax.set_xlabel("Compliance rate (count == 3)", fontsize=11)
ax.set_title("Constrained rewriting: per-transform compliance at high --n (50/80/100)", fontsize=12)
ax.axvline(0.95, color="gray", linestyle=":", alpha=0.5, label="95% threshold")
ax.grid(True, alpha=0.3, axis="x")
ax.legend(loc="lower right", frameon=False)
for i, (_, row) in enumerate(compliance_high.iterrows()):
    ax.text(row["rate"] + 0.01, i, f"{row['rate']:.0%}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "transform_compliance.png", dpi=120)
plt.show()

In [ ]:
# Which transforms are *entirely dropped* (count == 0) rather than under-produced?
# The JSON only contains transforms that produced at least 1 variant, so anything missing
# from a seed's JSON was silently lost.
EXPECTED_TRANSFORMS = {
    "polar_question", "polite_imperative", "indirect_need", "hedged_request",
    "conditional", "modal_would", "modal_should", "passive_voice",
    "negative_framing", "progressive_desire", "time_framed", "reason_added",
    "abbreviated_spoken", "emphatic", "confirmation_seeking",
}

missing_rows = []
for _, row in constrained.iterrows():
    present = set(json.loads(row["constrained_by_transform"]).keys())
    for tname in EXPECTED_TRANSFORMS - present:
        missing_rows.append({
            "seed_id": row["seed_id"],
            "seed": row["seed"],
            "syntactic_type": row["syntactic_type"],
            "utterance_length": row["utterance_length"],
            "n": row["n"],
            "transform_missing": tname,
        })
missing_df = pd.DataFrame(missing_rows)

print(f"Total instances where a transform produced 0 variants: {len(missing_df)}")
print("\nZero-output counts by transform:")
missing_df.groupby("transform_missing").size().sort_values(ascending=False)

In [ ]:
# Which syntactic types are the polar_question failures concentrated on?
polar_misses = missing_df[missing_df["transform_missing"] == "polar_question"]
print("polar_question misses by syntactic_type of the seed:")
print(polar_misses["syntactic_type"].value_counts())
print("\nSample seeds where polar_question failed:")
polar_misses[["seed", "syntactic_type", "n"]].drop_duplicates("seed").head(15)

**Key findings:**

- **Four transforms hit 100% compliance at high `--n`:** `time_framed`, `reason_added`, `hedged_request`, `confirmation_seeking`. These add content rather than rewriting structure, so the model always finds three ways to comply.
- **Two transforms underperform significantly:** `abbreviated_spoken` (67%) and `polar_question` (71%). These are structurally constrained — `abbreviated_spoken` wants a shorter form, which may collide with the original; `polar_question` fails systematically on seeds that already are polar questions (the model can't transform what's already the target form).
- **`polar_question` is also the most-dropped transform overall** (71 instances of producing zero variants across the sweep) — almost entirely on seeds whose `syntactic_type` is already `polar_question` or `wh_question`.

This is direct motivation for the constraint-customization future enhancement in the README: some transforms shouldn't be attempted on certain seed types.

---

## Section 5: Export summary CSV

Write a summary table that can be referenced directly from `volume_sweep.md` without rerunning the notebook.

In [ ]:
# Per-strategy summary: mean, median, p25, p75, min, max at each --n
summary_rows = []
for n in sorted(df["n"].unique()):
    for strat in STRATEGY_ORDER:
        sub = df[(df["n"] == n) & (df["strategy"] == strat)]["count"]
        summary_rows.append({
            "n": n,
            "strategy": strat,
            "mean": round(sub.mean(), 1),
            "median": int(sub.median()),
            "p25": int(np.percentile(sub, 25)),
            "p75": int(np.percentile(sub, 75)),
            "min": int(sub.min()),
            "max": int(sub.max()),
        })
summary = pd.DataFrame(summary_rows)
summary_path = RESULTS_DIR / "scaling_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"Wrote {summary_path}")
summary

In [ ]:
# Also export a compact version suitable for the generate_variants docstring
doc_table = df.pivot_table(index="n", columns="strategy", values="count", aggfunc="mean").round(0).astype(int)
doc_table = doc_table[STRATEGY_ORDER]
doc_table["total"] = doc_table.sum(axis=1)
print("Ready to paste into generate_variants docstring:\n")
print(doc_table.to_string())

---

## Summary of findings

1. **Constrained rewriting behaves exactly as designed.** Caps at ~14.5 for `--n < 45`, ~43.6 for `--n ≥ 50`, as predicted by the `min(3, n_per_strategy // 15)` formula.
2. **MLM substitution has a robust length-gated ceiling.** Short seeds cap around ~48 variants; medium seeds around ~75; long seeds are essentially unbounded within `--n ≤ 100` (median 100, p25 100).
3. **LLM paraphrasing has a content ceiling, not a mechanical one.** Short seeds plateau at ~71% efficiency; long seeds scale cleanly at ~97% efficiency.
4. **Constrained rewriting has two problem transforms at high `--n`:** `abbreviated_spoken` (67%) and `polar_question` (71%). The latter fails most often on seeds that are already interrogative.
5. **Total output at each `--n`** (full 117-seed set mean): ~30 / ~39 / ~58 / ~144 / ~187 / ~207 variants.
6. **Open question from this sweep:** do speech disfluencies (false starts, filled pauses) reduce LLM paraphrase yield when length is controlled? The two disfluency-tagged long seeds in this set landed at the bottom of the long-seed distribution. n=2 is not enough to answer, but worth revisiting.

Next step: draft `evaluation/results/volume_sweep.md` citing the plots and `scaling_summary.csv` produced by this notebook, then update the `generate_variants` docstring with the measured table and restore the scale guidance to the README.